In [1]:
"""
Script de détection automatique des capacités système pour modèles VLM
Génère automatiquement la configuration Docker optimale pour vLLM
"""
import subprocess
import platform
import re
import os
import json
from pathlib import Path

class SystemAnalyzer:
    def __init__(self):
        self.system_info = {
            'os': platform.system(),
            'ram_gb': 0,
            'gpu_available': False,
            'gpu_name': None,
            'vram_gb': 0,
            'cuda_available': False,
            'recommended_models': []
        }
    
    def get_ram(self):
        """Détecte la RAM système"""
        try:
            if self.system_info['os'] == 'Linux':
                # Linux
                with open('/proc/meminfo', 'r') as f:
                    meminfo = f.read()
                    match = re.search(r'MemTotal:\s+(\d+)', meminfo)
                    if match:
                        ram_kb = int(match.group(1))
                        self.system_info['ram_gb'] = round(ram_kb / (1024**2), 1)
            elif self.system_info['os'] == 'Darwin':
                # macOS
                result = subprocess.run(['sysctl', 'hw.memsize'], 
                                      capture_output=True, text=True)
                if result.returncode == 0:
                    ram_bytes = int(result.stdout.split()[1])
                    self.system_info['ram_gb'] = round(ram_bytes / (1024**3), 1)
            elif self.system_info['os'] == 'Windows':
                # Windows
                result = subprocess.run(['wmic', 'computersystem', 'get', 'totalphysicalmemory'],
                                      capture_output=True, text=True)
                if result.returncode == 0:
                    lines = result.stdout.strip().split('\n')
                    if len(lines) > 1:
                        ram_bytes = int(lines[1].strip())
                        self.system_info['ram_gb'] = round(ram_bytes / (1024**3), 1)
        except Exception as e:
            print(f"⚠️  Impossible de détecter la RAM: {e}")
            self.system_info['ram_gb'] = 8  # Valeur par défaut conservative
    
    def get_gpu_info(self):
        """Détecte les GPUs NVIDIA disponibles"""
        try:
            # Essayer nvidia-smi
            result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', 
                                   '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=5)
            
            if result.returncode == 0 and result.stdout.strip():
                lines = result.stdout.strip().split('\n')
                gpus = []
                for line in lines:
                    parts = line.split(',')
                    if len(parts) == 2:
                        gpu_name = parts[0].strip()
                        vram_mb = float(parts[1].strip())
                        gpus.append({
                            'name': gpu_name,
                            'vram_gb': round(vram_mb / 1024, 1)
                        })
                
                if gpus:
                    self.system_info['gpu_available'] = True
                    self.system_info['cuda_available'] = True
                    # Prendre le GPU avec le plus de VRAM
                    best_gpu = max(gpus, key=lambda x: x['vram_gb'])
                    self.system_info['gpu_name'] = best_gpu['name']
                    self.system_info['vram_gb'] = best_gpu['vram_gb']
                    
                    if len(gpus) > 1:
                        self.system_info['multi_gpu'] = True
                        self.system_info['gpu_count'] = len(gpus)
                        print(f"🎮 {len(gpus)} GPUs détectés, utilisation du meilleur: {best_gpu['name']}")
        
        except FileNotFoundError:
            print("⚠️  nvidia-smi non trouvé - Aucun GPU NVIDIA détecté")
        except subprocess.TimeoutExpired:
            print("⚠️  nvidia-smi timeout - Vérifiez vos drivers NVIDIA")
        except Exception as e:
            print(f"⚠️  Erreur détection GPU: {e}")
    
    def recommend_models(self):
        """Recommande les modèles compatibles"""
        vram = self.system_info['vram_gb']
        ram = self.system_info['ram_gb']
        
        models = []
        
        if self.system_info['gpu_available']:
            # Avec GPU
            if vram >= 24:
                models.append({
                    'name': 'Qwen2-VL-7B-Instruct',
                    'size': '7B',
                    'vram_required': '16-18 GB',
                    'performance': '⭐⭐⭐⭐⭐ Excellence',
                    'quantization': 'FP16',
                    'docker_config': {
                        'max_model_len': 8192,
                        'gpu_memory_utilization': 0.9,
                        'dtype': 'half'
                    }
                })
                models.append({
                    'name': 'Qwen3-VL-8B-Instruct',
                    'size': '8B',
                    'vram_required': '18-20 GB',
                    'performance': '⭐⭐⭐⭐⭐ Excellence',
                    'quantization': 'FP16',
                    'docker_config': {
                        'max_model_len': 8192,
                        'gpu_memory_utilization': 0.85,
                        'dtype': 'half'
                    }
                })
            
            if vram >= 12:
                models.append({
                    'name': 'Qwen3-VL-4B-Instruct',
                    'size': '4B',
                    'vram_required': '8-10 GB',
                    'performance': '⭐⭐⭐⭐ Très bon',
                    'quantization': 'FP16',
                    'recommended': True,
                    'docker_config': {
                        'max_model_len': 4096,
                        'gpu_memory_utilization': 0.85,
                        'dtype': 'half'
                    }
                })
            
            if vram >= 8:
                models.append({
                    'name': 'Qwen3-VL-2B-Instruct',
                    'size': '2B',
                    'vram_required': '5-6 GB',
                    'performance': '⭐⭐⭐ Bon',
                    'quantization': 'FP16',
                    'recommended': vram < 12,
                    'docker_config': {
                        'max_model_len': 4096,
                        'gpu_memory_utilization': 0.75,
                        'dtype': 'half'
                    }
                })
            
            if vram >= 6:
                models.append({
                    'name': 'Qwen3-VL-2B-Instruct',
                    'size': '2B',
                    'vram_required': '3-4 GB',
                    'performance': '⭐⭐⭐ Bon (quantifié)',
                    'quantization': 'INT8',
                    'recommended': vram < 8,
                    'docker_config': {
                        'max_model_len': 2048,
                        'gpu_memory_utilization': 0.7,
                        'dtype': 'auto',
                        'quantization': 'awq'
                    }
                })
        else:
            # CPU seulement
            print("\n⚠️  Aucun GPU détecté - vLLM nécessite un GPU NVIDIA pour fonctionner")
            print("💡 Alternatives suggérées:")
            print("   • Ollama (supporte CPU)")
            print("   • llama.cpp (supporte CPU)")
            print("   • OpenVINO (supporte CPU Intel)")
        
        self.system_info['recommended_models'] = models
    
    def analyze(self):
        """Analyse complète du système"""
        print("\n" + "="*70)
        print("🔍 ANALYSE DU SYSTÈME EN COURS...")
        print("="*70)
        
        print("\n📊 Détection de la configuration matérielle...")
        self.get_ram()
        self.get_gpu_info()
        
        print("\n" + "="*70)
        print("📋 RÉSULTATS DE L'ANALYSE")
        print("="*70)
        
        print(f"\n💻 Système d'exploitation: {self.system_info['os']}")
        print(f"🧠 RAM système: {self.system_info['ram_gb']} GB")
        
        if self.system_info['gpu_available']:
            print(f"🎮 GPU détecté: {self.system_info['gpu_name']}")
            print(f"💾 VRAM disponible: {self.system_info['vram_gb']} GB")
            print(f"✅ CUDA: Disponible")
        else:
            print("❌ GPU: Aucun GPU NVIDIA détecté")
            print("⚠️  vLLM nécessite un GPU NVIDIA avec CUDA")
        
        self.recommend_models()
        
        return self.system_info
    
    def print_recommendations(self):
        """Affiche les recommandations"""
        if not self.system_info['recommended_models']:
            print("\n❌ Aucun modèle vLLM compatible détecté")
            print("\n💡 Pour utiliser vLLM, vous avez besoin de:")
            print("   • Un GPU NVIDIA avec au moins 6 GB de VRAM")
            print("   • Drivers CUDA installés")
            print("   • nvidia-docker2 pour Docker")
            return None
        
        print("\n" + "="*70)
        print("🎯 MODÈLES COMPATIBLES AVEC VOTRE SYSTÈME")
        print("="*70)
        
        recommended_model = None
        for i, model in enumerate(self.system_info['recommended_models'], 1):
            is_recommended = model.get('recommended', False)
            prefix = "🌟 RECOMMANDÉ" if is_recommended else f"  {i}."
            
            print(f"\n{prefix}")
            print(f"   📦 Modèle: {model['name']}")
            print(f"   📏 Taille: {model['size']}")
            print(f"   💾 VRAM requise: {model['vram_required']}")
            print(f"   🎭 Quantization: {model['quantization']}")
            print(f"   ⚡ Performance: {model['performance']}")
            
            if is_recommended and recommended_model is None:
                recommended_model = model
        
        return recommended_model
    
    def generate_docker_files(self, model):
        """Génère les fichiers Docker optimisés"""
        if not model:
            return
        
        print("\n" + "="*70)
        print("🐋 GÉNÉRATION DES FICHIERS DOCKER")
        print("="*70)
        
        # Créer le dossier
        output_dir = Path("vllm-docker-config")
        output_dir.mkdir(exist_ok=True)
        
        # Générer Dockerfile
        dockerfile_content = self._generate_dockerfile(model)
        dockerfile_path = output_dir / "Dockerfile"
        with open(dockerfile_path, 'w') as f:
            f.write(dockerfile_content)
        print(f"\n✅ Dockerfile créé: {dockerfile_path}")
        
        # Générer docker-compose.yml
        compose_content = self._generate_compose(model)
        compose_path = output_dir / "docker-compose.yml"
        with open(compose_path, 'w') as f:
            f.write(compose_content)
        print(f"✅ docker-compose.yml créé: {compose_path}")
        
        # Générer .env
        env_content = self._generate_env(model)
        env_path = output_dir / ".env"
        with open(env_path, 'w') as f:
            f.write(env_content)
        print(f"✅ .env créé: {env_path}")
        
        # Générer README
        readme_content = self._generate_readme(model)
        readme_path = output_dir / "README.md"
        with open(readme_path, 'w') as f:
            f.write(readme_content)
        print(f"✅ README.md créé: {readme_path}")
        
        # Générer script de test
        test_content = self._generate_test_script(model)
        test_path = output_dir / "test_model.sh"
        with open(test_path, 'w') as f:
            f.write(test_content)
        test_path.chmod(0o755)
        print(f"✅ Script de test créé: {test_path}")
        
        print("\n" + "="*70)
        print("🎉 CONFIGURATION GÉNÉRÉE AVEC SUCCÈS!")
        print("="*70)
        print(f"\n📁 Tous les fichiers sont dans: {output_dir.absolute()}")
        print("\n🚀 Pour démarrer:")
        print(f"   cd {output_dir}")
        print("   docker-compose up -d")
        print("\n🧪 Pour tester:")
        print("   ./test_model.sh")
    
    def _generate_dockerfile(self, model):
        config = model['docker_config']
        return f"""FROM vllm/vllm-openai:latest

# Configuration optimisée pour {model['name']}
ENV MODEL_NAME="{model['name']}"
ENV HOST="0.0.0.0"
ENV PORT="8000"

# Optimisations spécifiques
ENV MAX_MODEL_LEN="{config['max_model_len']}"
ENV GPU_MEMORY_UTIL="{config['gpu_memory_utilization']}"
ENV DTYPE="{config['dtype']}"

# Dépendances supplémentaires
RUN pip install --no-cache-dir pillow requests

# Cache des modèles
RUN mkdir -p /root/.cache/huggingface

EXPOSE 8000

# Commande optimisée pour votre GPU ({self.system_info['vram_gb']} GB VRAM)
CMD vllm serve $MODEL_NAME \\
    --host $HOST \\
    --port $PORT \\
    --trust-remote-code \\
    --max-model-len $MAX_MODEL_LEN \\
    --gpu-memory-utilization $GPU_MEMORY_UTIL \\
    --dtype $DTYPE
"""
    
    def _generate_compose(self, model):
        return f"""version: '3.8'

services:
  vllm-server:
    build: .
    container_name: vllm-{model['size'].lower()}
    ports:
      - "8000:8000"
    environment:
      - MODEL_NAME={model['name']}
      - HUGGING_FACE_HUB_TOKEN=${{HUGGING_FACE_HUB_TOKEN}}
    volumes:
      - huggingface_cache:/root/.cache/huggingface
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    restart: unless-stopped
    shm_size: '8gb'
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

volumes:
  huggingface_cache:
"""
    
    def _generate_env(self, model):
        return f"""# Configuration générée automatiquement pour {model['name']}
# Système détecté: {self.system_info['gpu_name']} ({self.system_info['vram_gb']} GB VRAM)

# Token Hugging Face (optionnel pour modèles publics)
HUGGING_FACE_HUB_TOKEN=

# Configuration du modèle
MODEL_NAME={model['name']}
MAX_MODEL_LEN={model['docker_config']['max_model_len']}
GPU_MEMORY_UTIL={model['docker_config']['gpu_memory_utilization']}

# Configuration réseau
HOST=0.0.0.0
PORT=8000
"""
    
    def _generate_readme(self, model):
        return f"""# Configuration vLLM pour {model['name']}

Configuration automatiquement générée pour votre système.

## 🖥️ Système détecté

- **GPU**: {self.system_info['gpu_name']}
- **VRAM**: {self.system_info['vram_gb']} GB
- **RAM**: {self.system_info['ram_gb']} GB
- **OS**: {self.system_info['os']}

## 📦 Modèle recommandé

- **Nom**: {model['name']}
- **Taille**: {model['size']}
- **VRAM requise**: {model['vram_required']}
- **Performance attendue**: {model['performance']}
- **Quantization**: {model['quantization']}

## 🚀 Démarrage rapide

### 1. Vérifier le support GPU

```bash
docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi
```

### 2. Lancer le serveur

```bash
# Démarrer
docker-compose up -d

# Voir les logs
docker-compose logs -f

# Le serveur sera prêt quand vous verrez "Application startup complete"
```

### 3. Tester le modèle

```bash
./test_model.sh
```

Ou manuellement:

```bash
curl -X POST "http://localhost:8000/v1/chat/completions" \\
  -H "Content-Type: application/json" \\
  --data '{{
    "model": "{model['name']}",
    "messages": [
      {{
        "role": "user",
        "content": [
          {{
            "type": "text",
            "text": "Describe this image"
          }},
          {{
            "type": "image_url",
            "image_url": {{
              "url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
            }}
          }}
        ]
      }}
    ]
  }}'
```

## 📊 Performance attendue

Basé sur votre GPU ({self.system_info['gpu_name']}, {self.system_info['vram_gb']} GB VRAM):

- **Chargement du modèle**: 2-5 minutes (premier lancement)
- **Inférence**: {self._estimate_performance(model)}
- **Latence**: Faible à modérée

## 🛠️ Gestion

```bash
# Arrêter
docker-compose down

# Redémarrer
docker-compose restart

# Voir les logs
docker-compose logs -f

# Nettoyer tout (supprime aussi le cache)
docker-compose down -v
```

## ⚙️ Personnalisation

Éditez `.env` pour ajuster:
- `MAX_MODEL_LEN`: Longueur de contexte maximale
- `GPU_MEMORY_UTIL`: Utilisation de la VRAM (0.0-1.0)

## 🆘 Dépannage

### Le conteneur crash au démarrage

```bash
# Vérifier les logs détaillés
docker-compose logs vllm-server

# Réduire l'utilisation mémoire dans .env
GPU_MEMORY_UTIL=0.7
MAX_MODEL_LEN=2048
```

### Performance lente

- Vérifiez que le GPU est bien utilisé avec `nvidia-smi`
- Premier lancement = téléchargement du modèle (~{self._estimate_model_size(model)})
- Suivants = rapides grâce au cache

## 📚 API

API compatible OpenAI disponible sur `http://localhost:8000/v1/`

Endpoints:
- `/v1/chat/completions` - Chat avec images
- `/v1/models` - Liste des modèles
- `/health` - État du serveur
"""
    
    def _generate_test_script(self, model):
        return f"""#!/bin/bash

echo "🧪 Test du modèle {model['name']}"
echo "================================"

# Vérifier que le serveur est démarré
if ! docker ps | grep -q "vllm-{model['size'].lower()}"; then
    echo "❌ Le conteneur n'est pas démarré"
    echo "💡 Lancez d'abord: docker-compose up -d"
    exit 1
fi

# Attendre que le serveur soit prêt
echo "⏳ Attente du serveur..."
max_attempts=30
attempt=0

while [ $attempt -lt $max_attempts ]; do
    if curl -s http://localhost:8000/health > /dev/null 2>&1; then
        echo "✅ Serveur prêt!"
        break
    fi
    attempt=$((attempt + 1))
    echo "   Tentative $attempt/$max_attempts..."
    sleep 10
done

if [ $attempt -eq $max_attempts ]; then
    echo "❌ Le serveur ne répond pas après $max_attempts tentatives"
    echo "💡 Vérifiez les logs: docker-compose logs"
    exit 1
fi

# Test avec une image
echo ""
echo "🖼️  Test avec l'image de la Statue de la Liberté..."
echo ""

curl -X POST "http://localhost:8000/v1/chat/completions" \\
  -H "Content-Type: application/json" \\
  --data '{{
    "model": "{model['name']}",
    "messages": [
      {{
        "role": "user",
        "content": [
          {{
            "type": "text",
            "text": "Describe this image in one sentence."
          }},
          {{
            "type": "image_url",
            "image_url": {{
              "url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
            }}
          }}
        ]
      }}
    ],
    "max_tokens": 100
  }}' | python3 -m json.tool

echo ""
echo "✅ Test terminé!"
"""
    
    def _estimate_performance(self, model):
        vram = self.system_info['vram_gb']
        size = model['size']
        
        if '2B' in size:
            if vram >= 12:
                return "~20-30 tokens/sec (excellent)"
            elif vram >= 8:
                return "~15-25 tokens/sec (très bon)"
            else:
                return "~10-15 tokens/sec (bon)"
        elif '4B' in size:
            if vram >= 16:
                return "~15-20 tokens/sec (excellent)"
            elif vram >= 12:
                return "~10-15 tokens/sec (très bon)"
            else:
                return "~8-12 tokens/sec (bon)"
        elif '7B' or '8B' in size:
            if vram >= 24:
                return "~10-15 tokens/sec (excellent)"
            elif vram >= 16:
                return "~7-10 tokens/sec (bon)"
            else:
                return "~5-8 tokens/sec (acceptable)"
        
        return "Variable selon l'image"
    
    def _estimate_model_size(self, model):
        size = model['size']
        if '2B' in size:
            return "~5-6 GB"
        elif '4B' in size:
            return "~8-10 GB"
        elif '7B' or '8B' in size:
            return "~15-18 GB"
        return "Variable"


def main():
    print("""
    ╔══════════════════════════════════════════════════════════════════╗
    ║                                                                  ║
    ║     🚀 ANALYSEUR SYSTÈME POUR MODÈLES VLM (vLLM)               ║
    ║                                                                  ║
    ║     Détecte automatiquement votre configuration et génère       ║
    ║     les fichiers Docker optimaux pour votre système             ║
    ║                                                                  ║
    ╚══════════════════════════════════════════════════════════════════╝
    """)
    
    analyzer = SystemAnalyzer()
    
    # Analyse du système
    system_info = analyzer.analyze()
    
    # Recommandations
    recommended_model = analyzer.print_recommendations()
    
    # Génération des fichiers Docker
    if recommended_model:
        print("\n" + "="*70)
        response = input("\n❓ Voulez-vous générer les fichiers Docker pour le modèle recommandé? (o/n): ")
        
        if response.lower() in ['o', 'oui', 'y', 'yes']:
            analyzer.generate_docker_files(recommended_model)
            print("\n💡 Prochaines étapes:")
            print("   1. cd vllm-docker-config")
            print("   2. docker-compose up -d")
            print("   3. ./test_model.sh")
        else:
            print("\n👋 Génération annulée. Vous pouvez relancer ce script à tout moment.")
    
    # Sauvegarder l'analyse
    output_file = Path("system_analysis.json")
    with open(output_file, 'w') as f:
        json.dump(system_info, f, indent=2)
    print(f"\n💾 Analyse sauvegardée dans: {output_file.absolute()}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n👋 Analyse interrompue par l'utilisateur")
    except Exception as e:
        print(f"\n❌ Erreur: {e}")
        import traceback
        traceback.print_exc()


    ╔══════════════════════════════════════════════════════════════════╗
    ║                                                                  ║
    ║     🚀 ANALYSEUR SYSTÈME POUR MODÈLES VLM (vLLM)               ║
    ║                                                                  ║
    ║     Détecte automatiquement votre configuration et génère       ║
    ║     les fichiers Docker optimaux pour votre système             ║
    ║                                                                  ║
    ╚══════════════════════════════════════════════════════════════════╝
    

🔍 ANALYSE DU SYSTÈME EN COURS...

📊 Détection de la configuration matérielle...
⚠️  Impossible de détecter la RAM: invalid literal for int() with base 10: ''

📋 RÉSULTATS DE L'ANALYSE

💻 Système d'exploitation: Windows
🧠 RAM système: 8 GB
🎮 GPU détecté: NVIDIA GeForce GTX 1650
💾 VRAM disponible: 4.0 GB
✅ CUDA: Disponible

❌ Aucun modèle vLLM compatible détecté

💡 Pour utiliser vLLM, vous avez besoin de:
   • Un

In [ ]:
!pip install requests

In [ ]:
import requests

response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "Qwen/Qwen3-VL-4B-Instruct",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": "Que vois-tu sur cette image?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://example.com/image.jpg"
                    }
                }
            ]
        }],
        "max_tokens": 512,
        "temperature": 0.7
    }
)

print(response.json()["choices"][0]["message"]["content"])